# Final Project
#### Mateo Aguirre-Sandoval
#### December 12, 2025
#### CSCE 43233-0001

In [17]:
import os
import numpy as np
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

DATA_PATH = 'Sleep_health_and_lifestyle_dataset.csv'
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

In [18]:
#Task 1
print("Task 1: Load dataset")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("Dataset not found.")

df = pd.read_csv(DATA_PATH, na_values=['', 'NA', 'NaN'], keep_default_na=False)
print("Loaded dataset shape:", df.shape)

df.head().to_csv(os.path.join(OUTPUT_DIR, "head.csv"), index=False)
df.dtypes.to_csv(os.path.join(OUTPUT_DIR, "dtypes.csv"))

Task 1: Load dataset
Loaded dataset shape: (374, 13)


In [19]:
#Tasks 2-5
print("Task 3: Initial inspection")
print("Columns:", len(df.columns))
print("Rows:", len(df))
print("Data types (top):")
print(df.dtypes.head(20))
print("Missing values per column (descending):")
print(df.isna().sum().sort_values(ascending=False).head())

missing_frac = df.isna().mean()
cols_to_drop = missing_frac[missing_frac > 0.4].index.tolist()
print("Dropping columns with >40% missing:", cols_to_drop)

df = df.drop(columns=cols_to_drop)

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("Imputing numeric with median, categorical with mode...")

df[num_cols] = SimpleImputer(strategy="median").fit_transform(df[num_cols])
df[cat_cols] = SimpleImputer(strategy="most_frequent").fit_transform(df[cat_cols])

print("Missing values after imputation:", df.isna().sum().sum())

dups = df.duplicated().sum()
print("Number of duplicate rows:", dups)
if dups > 0:
    df = df.drop_duplicates()
    print("New shape after dropping duplicates:", df.shape)

Task 3: Initial inspection
Columns: 13
Rows: 374
Data types (top):
Person ID                    int64
Gender                      object
Age                          int64
Occupation                  object
Sleep Duration             float64
Quality of Sleep             int64
Physical Activity Level      int64
Stress Level                 int64
BMI Category                object
Blood Pressure              object
Heart Rate                   int64
Daily Steps                  int64
Sleep Disorder              object
dtype: object
Missing values per column (descending):
Person ID         0
Gender            0
Age               0
Occupation        0
Sleep Duration    0
dtype: int64
Dropping columns with >40% missing: []
Imputing numeric with median, categorical with mode...
Missing values after imputation: 0
Number of duplicate rows: 0


In [20]:
#Task 6
print("Task 6: Outlier detection & removal (IQR method)")
df_clean_unscaled = df.copy()
removed_outliers = pd.DataFrame()

for col in num_cols:
    Q1 = df_clean_unscaled[col].quantile(0.25)
    Q3 = df_clean_unscaled[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = df_clean_unscaled[(df_clean_unscaled[col] < lower) | (df_clean_unscaled[col] > upper)]
    if not outliers.empty:
        removed_outliers = pd.concat([removed_outliers, outliers])
        df_clean_unscaled = df_clean_unscaled[~((df_clean_unscaled[col] < lower) | (df_clean_unscaled[col] > upper))]

removed_outliers = removed_outliers.drop_duplicates()
print(f"Removed {len(removed_outliers)} outlier rows.")

removed_outliers.to_csv(os.path.join(OUTPUT_DIR, "removed_outliers.csv"), index=False)

print("Final shape after outlier removal:", df_clean_unscaled.shape)

Task 6: Outlier detection & removal (IQR method)
Removed 15 outlier rows.
Final shape after outlier removal: (359, 13)


In [21]:
#Task 7
print("Task 7: Scaling numerical features (StandardScaler)")
scaler = StandardScaler()
df_clean_scaled = df_clean_unscaled.copy()
df_clean_scaled[num_cols] = scaler.fit_transform(df_clean_scaled[num_cols])

df_clean_scaled.to_csv(os.path.join(OUTPUT_DIR, "sleep_health_clean_scaled.csv"), index=False)

Task 7: Scaling numerical features (StandardScaler)


In [22]:
#Task 8
print("Task 8: Summary statistics (unscaled)")
df_clean_unscaled.describe().T.to_csv(os.path.join(OUTPUT_DIR, "summary_stats.csv"))

Task 8: Summary statistics (unscaled)


In [23]:
#Task 9
print("Task 9: Plotting histograms")
top_numeric = df_clean_unscaled[num_cols].var().sort_values(ascending=False).head(6).index.tolist()

for col in top_numeric:
    plt.figure(figsize=(6,4))
    sns.histplot(df_clean_unscaled[col], kde=True)
    plt.title(f'Histogram: {col}')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, f'hist_{col}.png'))
    plt.close()

Task 9: Plotting histograms


In [24]:
#Task 10
print("Task 10: Correlation heatmap")
corr = df_clean_unscaled[num_cols].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "corr_heatmap.png"))
plt.close()


Task 10: Correlation heatmap


In [25]:
#Task 11
print("Task 11: Categorical feature frequency plots")
for col in cat_cols:
    print(f"Column: {col}")
    print(df_clean_unscaled[col].value_counts().head())
    plt.figure(figsize=(6,4))
    df_clean_unscaled[col].value_counts().plot(kind='bar', edgecolor='k')
    plt.title(f'Distribution of {col}')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, f'cat_{col}.png'))
    plt.close()

Task 11: Categorical feature frequency plots
Column: Gender
Gender
Female    180
Male      179
Name: count, dtype: int64
Column: Occupation
Occupation
Nurse       71
Doctor      67
Engineer    62
Lawyer      45
Teacher     39
Name: count, dtype: int64
Column: BMI Category
BMI Category
Normal           195
Overweight       145
Normal Weight     19
Name: count, dtype: int64
Column: Blood Pressure
Blood Pressure
130/85    99
140/95    65
125/80    65
120/80    45
115/75    32
Name: count, dtype: int64
Column: Sleep Disorder
Sleep Disorder
None           219
Insomnia        71
Sleep Apnea     69
Name: count, dtype: int64


In [26]:
#Task 12
print("Task 12: Random sampling (30%)")
sample_df_unscaled = df_clean_unscaled.sample(frac=0.30, random_state=42)
sample_df_unscaled.to_csv(os.path.join(OUTPUT_DIR, "sampled_fraction_10pct.csv"), index=False)
print("Sample shape:", sample_df_unscaled.shape)

Task 12: Random sampling (30%)
Sample shape: (108, 13)


In [27]:
#Task 13
print("Task 13: Creating SQLite database for Metabase...")
db_path = os.path.join(OUTPUT_DIR, "sleep_health.db")
conn = sqlite3.connect(db_path)

df_clean_unscaled.to_sql("sleep_health_cleaned_unscaled", conn, if_exists="replace", index=False)
df_clean_scaled.to_sql("sleep_health_cleaned_scaled", conn, if_exists="replace", index=False)
sample_df_unscaled.to_sql("sleep_health_sample_unscaled", conn, if_exists="replace", index=False)

summary_stats = df_clean_unscaled.describe().T.reset_index()
summary_stats = summary_stats.rename(columns={"index": "feature"})

summary_stats.to_sql("summary_statistics", conn, if_exists="replace", index=False)

print("SQLite DB created at", db_path)

Task 13: Creating SQLite database for Metabase...
SQLite DB created at outputs\sleep_health.db


In [28]:
#Task 14
print("Task 14: Boxplots + Violin plots")
for col in top_numeric:
    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1)
    sns.boxplot(x=df_clean_unscaled[col])
    plt.title(f'Boxplot {col}')
    
    plt.subplot(1,2,2)
    sns.violinplot(x=df_clean_unscaled[col])
    plt.title(f'Violin {col}')
    
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, f'box_violin_{col}.png'))
    plt.close()

Task 14: Boxplots + Violin plots


In [29]:
#Task 15
print("Task 15: Clustered heatmap")
sns.clustermap(df_clean_unscaled[num_cols].corr(), cmap="vlag", center=0)
plt.savefig(os.path.join(PLOTS_DIR, "clustered_heatmap.png"))
plt.close()

Task 15: Clustered heatmap


In [30]:
#Tasks 16-19
print("Task 16: Hypothesis testing setup")
print("Null H0: Physical activity is the same for disorder vs non-disorder")
print("Alternative H1: Means differ")

df_clean_unscaled['has_disorder'] = \
    df_clean_unscaled['Sleep Disorder'].apply(lambda x: 0 if x == "None" else 1)

activity_dis = df_clean_unscaled[df_clean_unscaled['has_disorder']==1]['Physical Activity Level']
activity_no  = df_clean_unscaled[df_clean_unscaled['has_disorder']==0]['Physical Activity Level']

print("Task 17: Performing Welch t-test")
t_stat, p_val = stats.ttest_ind(activity_dis, activity_no, equal_var=False)
print("Mean (Disorder):", activity_dis.mean())
print("Mean (No disorder):", activity_no.mean())
print("t-statistic:", t_stat)
print("p-value:", p_val)

print("Task 18: Decision at α=0.05")
alpha = 0.05
decision = "Reject H0" if p_val < alpha else "Fail to reject H0"
print("Decision:", decision)

print("Task 19: Confidence intervals (Welch + Bootstrap)")
mean_dis = activity_dis.mean()
mean_no = activity_no.mean()
diff = mean_dis - mean_no

se_dis = np.var(activity_dis, ddof=1) / len(activity_dis)
se_no  = np.var(activity_no, ddof=1) / len(activity_no)
se_diff = np.sqrt(se_dis + se_no)

df_welch = (se_dis + se_no)**2 / ((se_dis**2)/(len(activity_dis)-1) + (se_no**2)/(len(activity_no)-1))
t_crit = stats.t.ppf(0.975, df=df_welch)
ci_t = (diff - t_crit*se_diff, diff + t_crit*se_diff)

boot_diffs = []
for _ in range(5000):
    boot_dis = np.random.choice(activity_dis, size=len(activity_dis), replace=True)
    boot_no = np.random.choice(activity_no, size=len(activity_no), replace=True)
    boot_diffs.append(np.mean(boot_dis) - np.mean(boot_no))

boot_ci = (np.percentile(boot_diffs, 2.5), np.percentile(boot_diffs, 97.5))

print("t-based CI:", ci_t)
print("bootstrap CI:", boot_ci)

Task 16: Hypothesis testing setup
Null H0: Physical activity is the same for disorder vs non-disorder
Alternative H1: Means differ
Task 17: Performing Welch t-test
Mean (Disorder): 62.17857142857143
Mean (No disorder): 57.949771689497716
t-statistic: 1.8961956349803812
p-value: 0.05888902290989309
Task 18: Decision at α=0.05
Decision: Fail to reject H0
Task 19: Confidence intervals (Welch + Bootstrap)
t-based CI: (np.float64(-0.15981036514338243), np.float64(8.617409843290812))
bootstrap CI: (np.float64(-0.20114155251141685), np.float64(8.481559849967375))


In [31]:
#Task 20
print("Task 20: K-means clustering on PCA-reduced scaled data")

X = df_clean_scaled[num_cols]
pca = PCA(n_components=min(10, X.shape[1]))
X_pca = pca.fit_transform(X)

inertias = {}
silhouettes = {}

for k in range(2, 9):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_pca)
    inertias[k] = kmeans.inertia_

    try:
        silhouettes[k] = silhouette_score(X_pca, labels)
    except:
        silhouettes[k] = np.nan

print("Inertia by k:", inertias)
print("Silhouette by k:", silhouettes)

best_k = max(silhouettes, key=lambda k: silhouettes[k] if not np.isnan(silhouettes[k]) else -1)
print("Best k:", best_k)

final_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels = final_kmeans.fit_predict(X_pca)

clustered = pd.DataFrame(X_pca[:, :2], columns=['PCA 1', 'PCA 2'])
clustered['cluster'] = labels
clustered.to_csv(os.path.join(OUTPUT_DIR, "kmeans_pca_clusters.csv"), index=False)

print("Saving PCA cluster plot...")

loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PCA{i+1}" for i in range(pca.n_components_)],
    index=num_cols
)

loadings.to_csv(os.path.join(OUTPUT_DIR, "pca_loadings.csv"))

plt.figure(figsize=(8,6))
vibrant_palette = sns.color_palette("Set1", n_colors=best_k)

sns.scatterplot(
    data=clustered,
    x='PCA 1',
    y='PCA 2',
    hue='cluster',
    palette=vibrant_palette,
    s=60,
    edgecolor='black',
    linewidth=0.35
)

plt.title('KMeans Clusters (PCA 2D)', fontsize=14, fontweight='bold')
plt.xlabel("PCA 1 (Quality of Sleep, Sleep Duration, Stress Level, etc.)", fontsize=12)
plt.ylabel("PCA 2 (Physical Activity Level, Daily Steps, Age, etc.)", fontsize=12)
plt.legend(title="Cluster", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "kmeans_pca.png"), dpi=300)
plt.close()


Task 20: K-means clustering on PCA-reduced scaled data
Inertia by k: {2: 1901.7692810821336, 3: 1285.780472720748, 4: 866.988902587684, 5: 675.8998983155827, 6: 553.1540995477694, 7: 410.95097075608, 8: 312.2204820945726}
Silhouette by k: {2: 0.36867495794849386, 3: 0.4268640256154613, 4: 0.5146813613700648, 5: 0.5176750377261872, 6: 0.5452007688188383, 7: 0.5956142746668727, 8: 0.6248081583879661}
Best k: 8
Saving PCA cluster plot...


In [32]:
print("Saving analysis metrics to SQLite...")

analysis_metrics = pd.DataFrame([{
    "mean_disorder": mean_dis,
    "mean_no_disorder": mean_no,
    "difference_means": diff,
    "t_statistic": t_stat,
    "p_value": p_val,
    "alpha": 0.05,
    "decision": decision
}])

analysis_metrics.to_sql("analysis_metrics", conn, if_exists="replace", index=False)

ci_table = pd.DataFrame([
    {"type": "t_based", "ci_lower": ci_t[0], "ci_upper": ci_t[1]},
    {"type": "bootstrap", "ci_lower": boot_ci[0], "ci_upper": boot_ci[1]},
])
ci_table.to_sql("confidence_intervals", conn, if_exists="replace", index=False)

clustering_metrics = pd.DataFrame(
    [{"k": k, "inertia": inertias[k], "silhouette": silhouettes[k]} for k in inertias]
)
clustering_metrics.to_sql("kmeans_metrics", conn, if_exists="replace", index=False)

clustered.to_sql("kmeans_cluster_assignments", conn, if_exists="replace", index=False)

conn.close()



Saving analysis metrics to SQLite...
